# MandiMitra: Data Source Probe & Verification

## Objective
Programmatically verify three alternative data acquisition strategies to build a high-fidelity data pipeline for Onion daily market analysis across India.

## Audited Strategies
1. Programmatic Exploration of the CEDA Agri-Market API.
2. Structure inspection of the Kaggle historical mirror (2001–2026).
3. Parsing access limits of the official AGMARKNET api.data.gov.in portal.


In [2]:
import pandas as pd
import os

# Set target directory
parquet_dir = "../data/raw/parquet/"
sample_file = os.path.join(parquet_dir, "2025.parquet")

if os.path.exists(sample_file):
    # Read the compressed binary file via the new engine
    df = pd.read_parquet(sample_file)
    
    print("--- ARCHIVE AUDIT VERIFICATION ---")
    print(f"Total rows found in 2025 dataset: {len(df):,}")
    print(f"Columns inside file: {list(df.columns)}")
    
    # Isolate our anchor asset: Onion
    df['Commodity_Cleaned'] = df['Commodity'].astype(str).str.strip().str.lower()
    onion_data = df[df['Commodity_Cleaned'] == 'onion']
    
    print("\n--- COMMODITY VERIFICATION: ONION ---")
    print(f"Total recorded daily Onion observations: {len(onion_data):,}")
    if len(onion_data) > 0:
        print("\nSample Data Points:")
        print(onion_data[['State', 'District', 'Market', 'Modal_Price']].head().to_string())
else:
    print(f"File missing at path: {sample_file}. Please check that you copied the files into data/raw/parquet/")


--- ARCHIVE AUDIT VERIFICATION ---
Total rows found in 2025 dataset: 5,819,482
Columns inside file: ['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade', 'Arrival_Date', 'Min_Price', 'Max_Price', 'Modal_Price', 'Commodity_Code']

--- COMMODITY VERIFICATION: ONION ---
Total recorded daily Onion observations: 220,991

Sample Data Points:
          State      District                             Market  Modal_Price
5    Tamil Nadu  The Nilgiris         Gudalur (Uzhavar Sandhai )       4400.0
45   Tamil Nadu  Virudhunagar   Thalavaipuram (Uzhavar Sandhai )       4500.0
110  Tamil Nadu  The Nilgiris  Udhagamandalam (Uzhavar Sandhai )       4500.0
145  Tamil Nadu     Thanjavur       Thanjavur (Uzhavar Sandhai )       4600.0
174  Tamil Nadu     Thanjavur     Pattukottai (Uzhavar Sandhai )       5000.0


In [3]:
# Check row counts for our Phase 2 expansion commodities
expansion_crops = ['tomato', 'soybean']

print("--- EXPANSION COMMODITY AUDIT ---")
for crop in expansion_crops:
    crop_rows = df[df['Commodity'].astype(str).str.strip().str.lower() == crop]
    print(f"Total recorded daily {crop.capitalize()} observations in 2025: {len(crop_rows):,}")
    if len(crop_rows) > 0:
        print(f"   Sample States: {crop_rows['State'].unique()[:3]}")


--- EXPANSION COMMODITY AUDIT ---
Total recorded daily Tomato observations in 2025: 210,960
   Sample States: <StringArray>
['Tamil Nadu', 'Uttar Pradesh', 'Uttrakhand']
Length: 3, dtype: str
Total recorded daily Soybean observations in 2025: 0


In [4]:
# Look for any commodity containing 'soy' or 'bean' to unmask the true name
unique_commodities = df['Commodity'].astype(str).str.strip().str.lower().unique()
matching_names = [name for name in unique_commodities if 'soy' in name or 'bean' in name]

print("--- REVEALING TRUE COMMODITY SPELLINGS ---")
print(f"Found matching commodity names in the dataset: {matching_names}")


--- REVEALING TRUE COMMODITY SPELLINGS ---
Found matching commodity names in the dataset: ['beans', 'cluster beans', 'indian beans (seam)', 'black gram (urd beans)(whole)', 'soyabean', 'french beans (frasbean)', 'duster beans', 'guar seed (cluster beans seed)', 'bunch beans', 'surat beans (papadi)']


In [ ]:
# Query using the verified spelling string found in the data schema
soybean_rows = df[df['Commodity'].astype(str).str.strip().str.lower() == 'soyabean']

print("--- CORRECTED EXPANSION COMMODITY AUDIT ---")
print(f"Total recorded daily Soybean (Soyabean) observations in 2025: {len(soybean_rows):,}")
if len(soybean_rows) > 0:
    print(f"Sample States: {list(soybean_rows['State'].unique()[:4])}")


--- CORRECTED EXPANSION COMMODITY AUDIT ---
Total recorded daily Soybean (Soyabean) observations in 2025: 66,139
Sample States: ['Uttar Pradesh', 'Madhya Pradesh', 'Maharashtra', 'Gujarat']


In [6]:
import duckdb
from pathlib import Path

# Locate all original Parquet files
raw_dir = Path("../data/raw/parquet").resolve()
files = sorted(raw_dir.glob("*.parquet"))

print("Number of Parquet files:", len(files))
print("Files found:", [f.name for f in files])

if not files:
    raise FileNotFoundError("No Parquet files found.")

# Let DuckDB query the files directly
con = duckdb.connect()

parquet_path = (raw_dir / "*.parquet").as_posix()
parquet_path = parquet_path.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT *
    FROM read_parquet(
        '{parquet_path}',
        union_by_name = true,
        filename = true
    )
""")

# 1. Check the actual year-wise data coverage
year_report = con.execute("""
    SELECT
        regexp_extract(filename, '([0-9]{4})\\.parquet$', 1) AS year,
        COUNT(*) AS total_rows,
        MIN(TRY_CAST(Arrival_Date AS DATE)) AS first_date,
        MAX(TRY_CAST(Arrival_Date AS DATE)) AS last_date
    FROM raw
    GROUP BY year
    ORDER BY year
""").df()

print("\n--- YEAR-WISE INVENTORY ---")
display(year_report)

# 2. Find the actual commodity names used in the archive
crop_names = con.execute("""
    SELECT DISTINCT Commodity
    FROM raw
    WHERE
        contains(lower(CAST(Commodity AS VARCHAR)), 'onion')
        OR contains(lower(CAST(Commodity AS VARCHAR)), 'tomato')
        OR contains(lower(CAST(Commodity AS VARCHAR)), 'soy')
        OR contains(lower(CAST(Commodity AS VARCHAR)), 'cotton')
        OR contains(lower(CAST(Commodity AS VARCHAR)), 'grape')
        OR contains(lower(CAST(Commodity AS VARCHAR)), 'pomegranate')
    ORDER BY Commodity
""").df()

print("\n--- COMMODITY NAME VARIATIONS ---")
display(crop_names)

Number of Parquet files: 26
Files found: ['2001.parquet', '2002.parquet', '2003.parquet', '2004.parquet', '2005.parquet', '2006.parquet', '2007.parquet', '2008.parquet', '2009.parquet', '2010.parquet', '2011.parquet', '2012.parquet', '2013.parquet', '2014.parquet', '2015.parquet', '2016.parquet', '2017.parquet', '2018.parquet', '2019.parquet', '2020.parquet', '2021.parquet', '2022.parquet', '2023.parquet', '2024.parquet', '2025.parquet', '2026.parquet']

--- YEAR-WISE INVENTORY ---


,year,total_rows,first_date,last_date
0,2001,11603,2001-01-10,2001-12-31
1,2002,299930,2002-01-01,2002-12-31
2,2003,835030,2003-01-01,2003-12-31
3,2004,802338,2004-01-01,2004-12-31
4,2005,1073632,2005-01-01,2005-12-31
5,2006,1993050,2006-01-01,2006-12-31
6,2007,2378705,2007-01-01,2007-12-31
7,2008,2575390,2008-01-01,2008-12-31
8,2009,2483396,2009-01-01,2009-12-31
9,2010,2708429,2010-01-01,2010-12-31



--- COMMODITY NAME VARIATIONS ---


,Commodity
0,Cotton
1,Cotton Seed
2,Dry Grapes
3,Grapes
4,Onion
5,Onion Green
6,Pomegranate
7,Soyabean
8,Tomato


In [7]:
# MandiMitra: National Commodity Coverage Audit

import duckdb
from pathlib import Path

raw_dir = Path("../data/raw/parquet").resolve()
parquet_path = (raw_dir / "*.parquet").as_posix()

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT *
    FROM read_parquet(
        '{parquet_path}',
        union_by_name = true,
        filename = true
    )
""")

# Map actual source names to our project names
commodity_mapping = {
    "onion": "Onion",
    "tomato": "Tomato",
    "soyabean": "Soybean",
    "cotton": "Cotton",
    "grapes": "Grapes",
    "pomegranate": "Pomegranate"
}

mapping_sql = """
CASE lower(trim(CAST(Commodity AS VARCHAR)))
    WHEN 'onion' THEN 'Onion'
    WHEN 'tomato' THEN 'Tomato'
    WHEN 'soyabean' THEN 'Soybean'
    WHEN 'cotton' THEN 'Cotton'
    WHEN 'grapes' THEN 'Grapes'
    WHEN 'pomegranate' THEN 'Pomegranate'
    ELSE NULL
END
"""

# Keep only our six exact target commodities
con.execute(f"""
    CREATE OR REPLACE VIEW target_data AS
    SELECT *,
        {mapping_sql} AS Project_Commodity,
        TRY_CAST(Arrival_Date AS DATE) AS Parsed_Date
    FROM raw
    WHERE {mapping_sql} IS NOT NULL
""")

# 1. Coverage by commodity and year
coverage = con.execute("""
    SELECT
        Project_Commodity,
        regexp_extract(filename, '([0-9]{4})\\.parquet$', 1) AS File_Year,
        COUNT(*) AS Total_Rows,
        COUNT(DISTINCT State) AS Number_of_States,
        COUNT(DISTINCT Market) AS Number_of_Markets,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date,
        COUNT(*) FILTER (
            WHERE TRY_CAST(Modal_Price AS DOUBLE) IS NULL
        ) AS Missing_Modal_Prices
    FROM target_data
    GROUP BY Project_Commodity, File_Year
    ORDER BY Project_Commodity, File_Year
""").df()

print("--- SIX-COMMODITY HISTORICAL COVERAGE ---")
display(coverage)

# 2. Overall coverage for each commodity
summary = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Rows,
        COUNT(DISTINCT State) AS Number_of_States,
        COUNT(DISTINCT Market) AS Number_of_Markets,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date,
        COUNT(*) FILTER (
            WHERE TRY_CAST(Modal_Price AS DOUBLE) IS NULL
        ) AS Missing_Modal_Prices
    FROM target_data
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

print("\n--- OVERALL COMMODITY SUMMARY ---")
display(summary)

# 3. Investigate the 2025/2026 file overlap
overlap = con.execute("""
    SELECT
        filename,
        Parsed_Date,
        COUNT(*) AS Total_Rows
    FROM target_data
    WHERE Parsed_Date BETWEEN DATE '2025-12-30'
                          AND DATE '2025-12-31'
    GROUP BY filename, Parsed_Date
    ORDER BY filename, Parsed_Date
""").df()

print("\n--- DECEMBER 2025 FILE OVERLAP ---")
display(overlap)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SIX-COMMODITY HISTORICAL COVERAGE ---


,Project_Commodity,File_Year,Total_Rows,Number_of_States,Number_of_Markets,First_Date,Last_Date,Missing_Modal_Prices
0,Cotton,2001,305,7,10,2001-04-17,2001-12-31,0
1,Cotton,2002,6697,16,113,2002-01-01,2002-12-31,0
2,Cotton,2003,12937,15,184,2003-01-01,2003-12-31,0
3,Cotton,2004,15986,15,213,2004-01-01,2004-12-31,0
4,Cotton,2005,20424,15,371,2005-01-01,2005-12-31,0
...,...,...,...,...,...,...,...,...
151,Tomato,2022,189211,26,994,2022-01-01,2022-12-31,0
152,Tomato,2023,181567,27,1026,2023-01-01,2023-12-31,0
153,Tomato,2024,214864,25,1254,2024-01-01,2024-12-31,0
154,Tomato,2025,210960,25,1914,2025-01-01,2025-12-30,0



--- OVERALL COMMODITY SUMMARY ---


,Project_Commodity,Total_Rows,Number_of_States,Number_of_Markets,First_Date,Last_Date,Missing_Modal_Prices
0,Cotton,936032,24,1437,2001-04-17,2026-04-20,0
1,Grapes,369784,30,971,2001-01-27,2026-04-20,0
2,Onion,3168490,32,3180,2001-01-10,2026-04-20,0
3,Pomegranate,572365,25,854,2001-11-17,2026-04-20,0
4,Soybean,1192838,25,1446,2001-03-12,2026-04-20,0
5,Tomato,2818577,32,3033,2001-02-08,2026-04-20,0



--- DECEMBER 2025 FILE OVERLAP ---


,filename,Parsed_Date,Total_Rows
0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-30,1
1,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-30,602
2,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-31,672


In [8]:
duplicate_check = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date,
        COUNT(*) AS Record_Count
    FROM target_data
    GROUP BY
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date
    HAVING COUNT(*) > 1
    ORDER BY Record_Count DESC
    LIMIT 30
""").df()

print("Potential duplicate observations:")
display(duplicate_check)

duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS Duplicate_Groups,
        SUM(Record_Count - 1) AS Extra_Records
    FROM (
        SELECT COUNT(*) AS Record_Count
        FROM target_data
        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Potential duplicate observations:


,Project_Commodity,State,District,Market,Variety,Grade,Parsed_Date,Record_Count
0,Tomato,West Bengal,Bankura,Khatra APMC,Tomato,Non-FAQ,2025-12-28,2
1,Tomato,Uttar Pradesh,Khiri (Lakhimpur),Mohammdi APMC,Other,FAQ,2025-12-28,2
2,Onion,Tamil Nadu,Namakkal,Mohanur (Uzhavar Sandhai ) APMC,Bellary,Local,2025-12-28,2
3,Onion,Tamil Nadu,Thanjavur,Pattukottai (Uzhavar Sandhai ) APMC,Bellary,Local,2025-12-28,2
4,Pomegranate,Tamil Nadu,Perambalur,Perambalur (Uzhavar Sandhai ) APMC,Pomogranate,Local,2025-12-28,2
5,Tomato,Maharashtra,Raigad,Panvel APMC,Other,Local,2025-12-28,2
6,Onion,Tamil Nadu,Thirupathur,Natrampalli (Uzhavar Sandhai ) APMC,Bellary,Local,2025-12-28,2
7,Tomato,Tamil Nadu,Thanjavur,Pattukottai (Uzhavar Sandhai ) APMC,Deshi,Local,2025-12-28,2
8,Pomegranate,Tamil Nadu,Thanjavur,Thanjavur (Uzhavar Sandhai ) APMC,Pomogranate,Local,2025-12-28,2
9,Pomegranate,Tamil Nadu,Salem,Jalagandapuram (Uzhavar Sandhai ) APMC,Pomogranate,Local,2025-12-28,2


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Duplicate_Groups,Extra_Records
0,219,219.0


In [10]:
check = con.execute("""
    SELECT
        COUNT(*) AS Duplicate_Groups,
        SUM(Record_Count - 1) AS Extra_Records
    FROM (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date,
            COUNT(*) AS Record_Count
        FROM target_data
        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        HAVING COUNT(*) > 1
    )
""").df()

display(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Duplicate_Groups,Extra_Records
0,219,219.0


In [11]:
duplicate_details = con.execute("""
    WITH duplicates AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        FROM target_data
        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        HAVING COUNT(*) > 1
    )
    SELECT
        t.Project_Commodity,
        t.State,
        t.District,
        t.Market,
        t.Variety,
        t.Grade,
        t.Parsed_Date,
        t.Min_Price,
        t.Max_Price,
        t.Modal_Price,
        t.filename
    FROM target_data t
    INNER JOIN duplicates d
    USING (
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date
    )
    ORDER BY
        t.Project_Commodity,
        t.Parsed_Date,
        t.Market
""").df()

display(duplicate_details)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Variety,Grade,Parsed_Date,Min_Price,Max_Price,Modal_Price,filename
0,Cotton,Madhya Pradesh,Khargone,Badwaha APMC,Without Ginned Cotton,Non-FAQ,2025-12-28,7150.0,7150.0,7150.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
1,Cotton,Madhya Pradesh,Khargone,Badwaha APMC,Without Ginned Cotton,Non-FAQ,2025-12-28,7150.0,7150.0,7150.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
2,Cotton,Madhya Pradesh,Khargone,Bhikangaon APMC,Without Ginned Cotton,FAQ,2025-12-28,5400.0,7720.0,6000.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
3,Cotton,Madhya Pradesh,Khargone,Bhikangaon APMC,Without Ginned Cotton,FAQ,2025-12-28,5400.0,7720.0,6000.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
4,Cotton,Maharashtra,Nagpur,Bhiwapur APMC,Other,Non-FAQ,2025-12-28,7450.0,7620.0,7535.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
...,...,...,...,...,...,...,...,...,...,...,...
433,Tomato,Uttarakhand,Dehradoon,Vikasnagar APMC,Other,Medium,2025-12-28,1800.0,2100.0,2100.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
434,Tomato,Tamil Nadu,Cuddalore,Viruthachalam (Uzhavar Sandhai ) APMC,Deshi,Local,2025-12-28,6000.0,6000.0,6000.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
435,Tomato,Tamil Nadu,Cuddalore,Viruthachalam (Uzhavar Sandhai ) APMC,Deshi,Local,2025-12-28,6000.0,6000.0,6000.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...
436,Tomato,Punjab,Amritsar,Rayya APMC,Other,FAQ,2025-12-30,1250.0,1250.0,1250.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...


In [12]:
exact_duplicate_check = con.execute("""
    WITH duplicate_groups AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        FROM target_data
        GROUP BY
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS Rows_In_Duplicate_Groups,
        COUNT(DISTINCT filename) AS Source_Files,
        COUNT(*) FILTER (
            WHERE Min_Price IS NULL
               OR Max_Price IS NULL
               OR Modal_Price IS NULL
        ) AS Rows_With_Missing_Prices,
        COUNT(*) FILTER (
            WHERE TRY_CAST(Min_Price AS DOUBLE)
                  = TRY_CAST(Max_Price AS DOUBLE)
              AND TRY_CAST(Max_Price AS DOUBLE)
                  = TRY_CAST(Modal_Price AS DOUBLE)
        ) AS Rows_Where_All_Prices_Match
    FROM target_data t
    INNER JOIN duplicate_groups d
    USING (
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date
    )
""").df()

display(exact_duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Rows_In_Duplicate_Groups,Source_Files,Rows_With_Missing_Prices,Rows_Where_All_Prices_Match
0,438,2,0,100


In [13]:
duplicate_diagnostic = con.execute("""
    WITH grouped AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date,
            COUNT(*) AS Record_Count,
            COUNT(DISTINCT Min_Price) AS Unique_Min_Prices,
            COUNT(DISTINCT Max_Price) AS Unique_Max_Prices,
            COUNT(DISTINCT Modal_Price) AS Unique_Modal_Prices,
            COUNT(*) FILTER (
                WHERE Min_Price IS NULL
                   OR Max_Price IS NULL
                   OR Modal_Price IS NULL
            ) AS Missing_Price_Rows
        FROM target_data
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS Duplicate_Groups,
        SUM(Record_Count) AS Rows_In_Duplicate_Groups,
        SUM(Record_Count - 1) AS Extra_Records,
        COUNT(*) FILTER (
            WHERE Unique_Min_Prices = 1
              AND Unique_Max_Prices = 1
              AND Unique_Modal_Prices = 1
        ) AS Groups_With_Identical_Prices,
        COUNT(*) FILTER (
            WHERE Unique_Min_Prices > 1
               OR Unique_Max_Prices > 1
               OR Unique_Modal_Prices > 1
        ) AS Groups_With_Different_Prices
    FROM grouped
""").df()

display(duplicate_diagnostic)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Duplicate_Groups,Rows_In_Duplicate_Groups,Extra_Records,Groups_With_Identical_Prices,Groups_With_Different_Prices
0,219,438.0,219.0,219,0


In [14]:
source_check = con.execute("""
    SELECT
        regexp_extract(filename, '([0-9]{4})\\.parquet$', 1) AS File_Year,
        COUNT(*) AS Duplicate_Rows
    FROM target_data
    WHERE Parsed_Date = DATE '2025-12-28'
    GROUP BY File_Year
    ORDER BY File_Year
""").df()

display(source_check)

,File_Year,Duplicate_Rows
0,2025,579


In [15]:
duplicate_source = con.execute("""
    WITH grouped AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Parsed_Date,
            COUNT(*) AS Record_Count
        FROM target_data
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT
        regexp_extract(t.filename, '([0-9]{4})\\.parquet$', 1) AS File_Year,
        COUNT(*) AS Rows_In_Duplicate_Groups,
        COUNT(DISTINCT t.Project_Commodity) AS Commodities_Affected
    FROM target_data t
    INNER JOIN grouped g
    ON t.Project_Commodity = g.Project_Commodity
       AND t.State = g.State
       AND t.District = g.District
       AND t.Market = g.Market
       AND t.Variety = g.Variety
       AND t.Grade = g.Grade
       AND t.Parsed_Date = g.Parsed_Date
    GROUP BY File_Year
    ORDER BY File_Year
""").df()

display(duplicate_source)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,File_Year,Rows_In_Duplicate_Groups,Commodities_Affected
0,2025,437,6
1,2026,1,1


In [16]:
boundary_check = con.execute("""
    SELECT
        filename,
        Parsed_Date,
        COUNT(*) AS Total_Rows,
        COUNT(DISTINCT Project_Commodity) AS Commodities
    FROM target_data
    WHERE Parsed_Date BETWEEN DATE '2025-12-29'
                          AND DATE '2026-01-02'
    GROUP BY filename, Parsed_Date
    ORDER BY Parsed_Date, filename
""").df()

display(boundary_check)

,filename,Parsed_Date,Total_Rows,Commodities
0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-29,581,6
1,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-30,1,1
2,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-30,602,6
3,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2025-12-31,672,6
4,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2026-01-01,523,6
5,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,2026-01-02,345,6


In [17]:
overlap_check = con.execute("""
WITH old_file AS (
    SELECT *
    FROM target_data
    WHERE regexp_extract(filename, '([0-9]{4})\\.parquet$', 1) = '2025'
      AND Parsed_Date = DATE '2025-12-30'
),
new_file AS (
    SELECT *
    FROM target_data
    WHERE regexp_extract(filename, '([0-9]{4})\\.parquet$', 1) = '2026'
      AND Parsed_Date = DATE '2025-12-30'
)
SELECT
    n.Project_Commodity,
    n.State,
    n.District,
    n.Market,
    n.Variety,
    n.Grade,
    n.Parsed_Date,
    o.Modal_Price AS Price_In_2025_File,
    n.Modal_Price AS Price_In_2026_File,
    o.Min_Price AS Min_In_2025_File,
    n.Min_Price AS Min_In_2026_File,
    o.Max_Price AS Max_In_2025_File,
    n.Max_Price AS Max_In_2026_File
FROM new_file n
JOIN old_file o
    ON n.Project_Commodity IS NOT DISTINCT FROM o.Project_Commodity
    AND n.State IS NOT DISTINCT FROM o.State
    AND n.District IS NOT DISTINCT FROM o.District
    AND n.Market IS NOT DISTINCT FROM o.Market
    AND n.Variety IS NOT DISTINCT FROM o.Variety
    AND n.Grade IS NOT DISTINCT FROM o.Grade
    AND n.Parsed_Date = o.Parsed_Date
""").df()

display(overlap_check)

,Project_Commodity,State,District,Market,Variety,Grade,Parsed_Date,Price_In_2025_File,Price_In_2026_File,Min_In_2025_File,Min_In_2026_File,Max_In_2025_File,Max_In_2026_File
0,Tomato,Punjab,Amritsar,Rayya APMC,Other,FAQ,2025-12-30,1250.0,1250.0,1250.0,1250.0,1250.0,1250.0


In [18]:
price_quality = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Min_Price < 0
               OR Max_Price < 0
               OR Modal_Price < 0
        ) AS Negative_Price_Records,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Min_Greater_Than_Max,

        COUNT(*) FILTER (
            WHERE Modal_Price < Min_Price
               OR Modal_Price > Max_Price
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE Min_Price IS NULL
               OR Max_Price IS NULL
               OR Modal_Price IS NULL
        ) AS Missing_Price_Records

    FROM target_data
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(price_quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Negative_Price_Records,Min_Greater_Than_Max,Modal_Outside_Range,Missing_Price_Records
0,Cotton,936032,0,97,3707,1
1,Grapes,369784,0,132,2347,0
2,Onion,3168490,0,2133,125287,1
3,Pomegranate,572365,0,133,6439,0
4,Soybean,1192838,0,255,3263,0
5,Tomato,2818577,0,2309,86320,0


In [19]:
invalid_samples = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        Variety,
        Grade,
        Parsed_Date,
        Min_Price,
        Modal_Price,
        Max_Price,
        filename,
        CASE
            WHEN Min_Price > Max_Price
                THEN 'Minimum greater than maximum'
            WHEN Modal_Price < Min_Price
                THEN 'Modal below minimum'
            WHEN Modal_Price > Max_Price
                THEN 'Modal above maximum'
        END AS Problem
    FROM target_data
    WHERE
        Min_Price > Max_Price
        OR Modal_Price < Min_Price
        OR Modal_Price > Max_Price
    ORDER BY Project_Commodity, Parsed_Date
    LIMIT 30
""").df()

display(invalid_samples)

,Project_Commodity,State,District,Market,Variety,Grade,Parsed_Date,Min_Price,Modal_Price,Max_Price,filename,Problem
0,Cotton,Bihar,Siwan,Siwan,Other,FAQ,2002-06-14,0.0,1500.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
1,Cotton,Odisha,Kandhamal,Tikabali,Other,FAQ,2002-06-14,0.0,1450.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
2,Cotton,Telangana,Karimnagar,Jammikunta,Other,FAQ,2002-06-14,0.0,1540.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
3,Cotton,Telangana,Khammam,Khammam,Other,FAQ,2002-06-14,0.0,1730.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
4,Cotton,Telangana,Warangal,Warangal,Other,FAQ,2002-06-14,0.0,1600.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
5,Cotton,Telangana,Karimnagar,Karimnagar,Other,FAQ,2002-06-15,0.0,1560.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
6,Cotton,Telangana,Khammam,Khammam,Other,FAQ,2002-06-15,0.0,1700.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
7,Cotton,Telangana,Warangal,Warangal,Other,FAQ,2002-06-15,0.0,1600.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
8,Cotton,Telangana,Karimnagar,Jammikunta,Other,FAQ,2002-06-17,0.0,1530.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum
9,Cotton,Telangana,Khammam,Khammam,Other,FAQ,2002-06-17,0.0,1700.0,0.0,C:/Users/Lenovo/Documents/GrittingCode/MandiMi...,Modal above maximum


In [20]:
zero_price_audit = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Min_Price = 0 AND Max_Price = 0
        ) AS Both_Min_Max_Zero,

        COUNT(*) FILTER (
            WHERE Min_Price = 0 AND Max_Price = 0
              AND Modal_Price > 0
        ) AS Both_Zero_Modal_Positive,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
               OR Max_Price = 0
               OR Modal_Price = 0
        ) AS Any_Zero_Price,

        MIN(Parsed_Date) FILTER (
            WHERE Min_Price = 0 AND Max_Price = 0
        ) AS First_Occurrence,

        MAX(Parsed_Date) FILTER (
            WHERE Min_Price = 0 AND Max_Price = 0
        ) AS Last_Occurrence

    FROM target_data
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(zero_price_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,Total_Records,Both_Min_Max_Zero,Both_Zero_Modal_Positive,Any_Zero_Price,First_Occurrence,Last_Occurrence
0,Cotton,936032,3609,3609,3790,2002-06-14,2025-10-23
1,Grapes,369784,2215,2215,3006,2003-02-06,2023-05-10
2,Onion,3168490,122479,122479,129314,2002-08-09,2025-10-04
3,Pomegranate,572365,6306,6306,7504,2008-05-07,2023-04-26
4,Soybean,1192838,494,494,2775,2002-07-19,2025-11-03
5,Tomato,2818577,84010,84010,88060,2002-07-23,2025-09-15


In [21]:
zero_by_year = con.execute("""
    SELECT
        Project_Commodity,
        YEAR(Parsed_Date) AS Year,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
              AND Max_Price = 0
              AND Modal_Price > 0
        ) AS Zero_Range_Records,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE Min_Price = 0
                  AND Max_Price = 0
                  AND Modal_Price > 0
            ) / COUNT(*),
            2
        ) AS Affected_Percentage

    FROM target_data
    GROUP BY Project_Commodity, YEAR(Parsed_Date)
    ORDER BY Project_Commodity, Year
""").df()

display(zero_by_year)

,Project_Commodity,Year,Total_Records,Zero_Range_Records,Affected_Percentage
0,Cotton,2001,305,0,0.00
1,Cotton,2002,6697,687,10.26
2,Cotton,2003,12937,1286,9.94
3,Cotton,2004,15986,1039,6.50
4,Cotton,2005,20424,0,0.00
...,...,...,...,...,...
151,Tomato,2022,189211,127,0.07
152,Tomato,2023,181567,1,0.00
153,Tomato,2024,214864,1,0.00
154,Tomato,2025,211397,1,0.00


In [22]:
recent_quality = con.execute("""
    SELECT
        Project_Commodity,
        COUNT(*) AS Total_Records,

        COUNT(*) FILTER (
            WHERE Min_Price = 0
              AND Max_Price = 0
              AND Modal_Price > 0
        ) AS Zero_Range_Records,

        COUNT(*) FILTER (
            WHERE Min_Price > Max_Price
        ) AS Invalid_Ranges,

        COUNT(*) FILTER (
            WHERE Modal_Price < Min_Price
               OR Modal_Price > Max_Price
        ) AS Modal_Outside_Range,

        COUNT(*) FILTER (
            WHERE Modal_Price IS NULL
        ) AS Missing_Modal,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE Min_Price = 0
                  AND Max_Price = 0
                  AND Modal_Price > 0
            ) / COUNT(*),
            3
        ) AS Zero_Range_Percentage

    FROM target_data
    WHERE YEAR(Parsed_Date) BETWEEN 2021 AND 2026
    GROUP BY Project_Commodity
    ORDER BY Project_Commodity
""").df()

display(recent_quality)

,Project_Commodity,Total_Records,Zero_Range_Records,Invalid_Ranges,Modal_Outside_Range,Missing_Modal,Zero_Range_Percentage
0,Cotton,161822,409,16,426,0,0.253
1,Grapes,108142,89,65,154,0,0.082
2,Onion,1034345,152,1045,1872,0,0.015
3,Pomegranate,238815,96,60,156,0,0.040
4,Soybean,324371,97,70,2681,0,0.030
5,Tomato,979831,161,1374,1536,0,0.016


In [23]:
reporting_audit = con.execute("""
    SELECT
        Project_Commodity,
        YEAR(Parsed_Date) AS Year,
        COUNT(DISTINCT Parsed_Date) AS Reporting_Days,
        COUNT(DISTINCT Market) AS Markets,
        COUNT(*) AS Total_Records
    FROM target_data
    GROUP BY Project_Commodity, YEAR(Parsed_Date)
    ORDER BY Project_Commodity, Year
""").df()

display(reporting_audit)

,Project_Commodity,Year,Reporting_Days,Markets,Total_Records
0,Cotton,2001,169,10,305
1,Cotton,2002,340,113,6697
2,Cotton,2003,348,184,12937
3,Cotton,2004,360,213,15986
4,Cotton,2005,361,371,20424
...,...,...,...,...,...
151,Tomato,2022,365,994,189211
152,Tomato,2023,365,1026,181567
153,Tomato,2024,366,1254,214864
154,Tomato,2025,342,1924,211397


In [24]:
market_history = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,

        COUNT(*) AS Total_Records,

        COUNT(DISTINCT Parsed_Date) AS Reporting_Days,

        MIN(Parsed_Date) AS First_Observation,

        MAX(Parsed_Date) AS Last_Observation

    FROM target_data
    GROUP BY
        Project_Commodity,
        State,
        District,
        Market

    ORDER BY
        Project_Commodity,
        Reporting_Days DESC
""").df()

display(market_history)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Total_Records,Reporting_Days,First_Observation,Last_Observation
0,Cotton,Gujarat,Rajkot,Rajkot,5737,5736,2002-04-02,2025-11-05
1,Cotton,Gujarat,Amreli,Amreli,5634,5634,2003-04-05,2025-11-04
2,Cotton,Andhra Pradesh,Kurnool,Adoni,6412,5584,2002-07-01,2025-11-05
3,Cotton,Karnataka,Raichur,Raichur,5763,5282,2002-02-26,2025-11-04
4,Cotton,Karnataka,Bijapur,Bijapur,6112,5174,2002-03-06,2025-03-01
...,...,...,...,...,...,...,...,...
11048,Tomato,Arunachal Pradesh,East Siang,Pasighat,1,1,2005-04-08,2005-04-08
11049,Tomato,Andhra Pradesh,East Godavari,Kothapeta,1,1,2008-02-02,2008-02-02
11050,Tomato,Maharashtra,Pune,Junnar (Alephata),1,1,2011-06-03,2011-06-03
11051,Tomato,Odisha,Kandhamal,Tikabali APMC,1,1,2026-01-03,2026-01-03


In [25]:
market_summary = (
    market_history
    .groupby("Project_Commodity")
    .agg(
        Total_Markets=("Market", "count"),
        Median_Reporting_Days=("Reporting_Days", "median"),
        Markets_Over_3_Years=(
            "Reporting_Days",
            lambda x: (x >= 1095).sum()
        ),
        Markets_Over_1_Year=(
            "Reporting_Days",
            lambda x: (x >= 365).sum()
        )
    )
    .reset_index()
)

display(market_summary)

,Project_Commodity,Total_Markets,Median_Reporting_Days,Markets_Over_3_Years,Markets_Over_1_Year
0,Cotton,1459,86.0,276,473
1,Grapes,980,43.5,100,258
2,Onion,3223,91.0,856,1323
3,Pomegranate,859,66.0,204,327
4,Soybean,1457,42.0,336,493
5,Tomato,3075,106.0,817,1208


In [26]:
top_markets = con.execute("""
    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        COUNT(DISTINCT Parsed_Date) AS Reporting_Days,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date
    FROM target_data
    GROUP BY ALL
    HAVING COUNT(DISTINCT Parsed_Date) >= 1000
    ORDER BY Project_Commodity, Reporting_Days DESC
""").df()

display(
    top_markets.groupby("Project_Commodity")
    .head(3)
    .reset_index(drop=True)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Reporting_Days,First_Date,Last_Date
0,Cotton,Gujarat,Rajkot,Rajkot,5736,2002-04-02,2025-11-05
1,Cotton,Gujarat,Amreli,Amreli,5634,2003-04-05,2025-11-04
2,Cotton,Andhra Pradesh,Kurnool,Adoni,5584,2002-07-01,2025-11-05
3,Grapes,Karnataka,Bangalore,"Binny Mill (F&V), Bangalore",5369,2001-11-17,2025-10-30
4,Grapes,Maharashtra,Nagpur,Nagpur,4692,2001-05-26,2025-11-04
5,Grapes,NCT of Delhi,Delhi,Azadpur,4542,2005-08-11,2025-11-01
6,Onion,Uttar Pradesh,Bareilly,Bareilly,7372,2002-08-14,2025-11-04
7,Onion,Odisha,Bargarh,Bargarh,7357,2003-02-04,2025-11-03
8,Onion,Uttar Pradesh,Rampur,Rampur,7269,2002-09-09,2025-11-02
9,Pomegranate,Karnataka,Bangalore,"Binny Mill (F&V), Bangalore",5717,2001-11-17,2025-10-31


In [29]:
gap_audit = con.execute("""
    WITH market_dates AS (
        SELECT DISTINCT
            Project_Commodity,
            State,
            District,
            Market,
            Parsed_Date
        FROM target_data
    ),

    date_gaps AS (
        SELECT
            Project_Commodity,
            State,
            District,
            Market,
            Parsed_Date,
            LAG(Parsed_Date) OVER (
                PARTITION BY
                    Project_Commodity,
                    State,
                    District,
                    Market
                ORDER BY Parsed_Date
            ) AS Previous_Date
        FROM market_dates
    )

    SELECT
        Project_Commodity,
        State,
        District,
        Market,
        COUNT(*) AS Reporting_Days,
        MAX(date_diff('day', Previous_Date, Parsed_Date))
            AS Largest_Gap_Days,
        COUNT(*) FILTER (
            WHERE date_diff('day', Previous_Date, Parsed_Date) > 14
        ) AS Gaps_Over_14_Days,
        COUNT(*) FILTER (
            WHERE date_diff('day', Previous_Date, Parsed_Date) > 30
        ) AS Gaps_Over_30_Days

    FROM date_gaps
    GROUP BY Project_Commodity, State, District, Market
    HAVING COUNT(*) >= 1000
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY Project_Commodity
        ORDER BY COUNT(*) DESC
    ) <= 3
    ORDER BY Project_Commodity, Reporting_Days DESC
""").df()

display(gap_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Project_Commodity,State,District,Market,Reporting_Days,Largest_Gap_Days,Gaps_Over_14_Days,Gaps_Over_30_Days
0,Cotton,Gujarat,Rajkot,Rajkot,5736,75,11,6
1,Cotton,Gujarat,Amreli,Amreli,5634,87,12,5
2,Cotton,Andhra Pradesh,Kurnool,Adoni,5584,734,11,3
3,Grapes,Karnataka,Bangalore,"Binny Mill (F&V), Bangalore",5369,291,21,9
4,Grapes,Maharashtra,Nagpur,Nagpur,4692,165,34,12
5,Grapes,NCT of Delhi,Delhi,Azadpur,4542,366,15,12
6,Onion,Uttar Pradesh,Bareilly,Bareilly,7372,103,6,2
7,Onion,Odisha,Bargarh,Bargarh,7357,119,9,5
8,Onion,Uttar Pradesh,Rampur,Rampur,7269,34,6,1
9,Pomegranate,Karnataka,Bangalore,"Binny Mill (F&V), Bangalore",5717,74,15,4
